# UCI Bank Marketing Dataset - Exploratory Data Analysis (EDA)

Welcome to the **UCI Bank Marketing Machine Learning Project**!

### Objective:
The goal of this project is to predict whether a bank client will subscribe to a term deposit (`y = yes / no`) based on marketing campaign phone calls and client demographic data.

### Dataset Details:
- **Source**: [UCI Machine Learning Repository (ID 222)](https://archive.ics.uci.edu/dataset/222/bank+marketing)
- **Features**: Client demographics (`age`, `job`, `marital`, `education`), financial history (`default`, `balance`, `housing`, `loan`), campaign contact info (`contact`, `day`, `month`, `duration`, `campaign`, `pdays`, `previous`, `poutcome`).
- **Target**: `y` (binary: `'yes'` or `'no'`)

In [ ]:
# 1. Setup paths and imports
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
%matplotlib inline

from src.data.load_data import load_bank_marketing_data
print("Imports successful!")

In [ ]:
# 2. Load the dataset (cached automatically to data/raw/)
X, y = load_bank_marketing_data()
df = X.copy()
df['target'] = y

print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Features: {X.shape[1]}")
df.head()

In [ ]:
# 3. Target Distribution (Class Imbalance Check)
counts = df['target'].value_counts()
percentages = df['target'].value_counts(normalize=True) * 100

plt.figure(figsize=(6, 4))
ax = sns.barplot(x=counts.index, y=counts.values, palette="viridis")
plt.title("Target Distribution (Subscription to Term Deposit)", fontsize=14)
plt.xlabel("Subscribed? (y)")
plt.ylabel("Count")

for i, p in enumerate(percentages):
    ax.annotate(f"{counts.iloc[i]:,} ({p:.1f}%)", (i, counts.iloc[i] / 2), ha='center', color='white', fontweight='bold')
plt.show()

In [ ]:
# 4. Summary Statistics for Numeric Features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
df[numeric_cols].describe()

In [ ]:
# 5. Check Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Percentage (%)": missing_pct})
missing_df[missing_df["Missing Count"] > 0]

In [ ]:
# 6. Categorical Features Distribution
cat_cols = ['job', 'marital', 'education', 'housing', 'loan']

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    sns.countplot(data=df, y=col, hue='target', ax=axes[idx], palette="coolwarm")
    axes[idx].set_title(f"Distribution of {col} by Target")
    axes[idx].set_xlabel("Count")
    axes[idx].set_ylabel(col)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

### Key Takeaways & Next Steps:
1. **Imbalanced Target**: Term deposit subscriptions are only ~11.7% of all contacts. Metrics like **Precision, Recall, F1-Score, and ROC-AUC** are much more informative than simple Accuracy.
2. **Data Pipeline**: Preprocessing will use `ColumnTransformer` with `StandardScaler` for numeric columns and `OneHotEncoder` for categorical columns.
3. **Models**: Test baseline Logistic Regression with `class_weight='balanced'`, followed by ensemble models (Random Forest, Gradient Boosting).

To run the baseline models now, run in terminal:
```bash
python -m src.models.train
```